In [4]:
import chromadb
chroma_client=chromadb.PersistentClient(path="./../../data/chromadb")

** Sqlite3 is used to store metadata **

In [13]:
collection = chroma_client.create_collection(name="my_collection1")


In [6]:
collection.add(
    ids=["1", "2"],
    embeddings=[[0.1, 0.2, 0.3], [0.4, 0.5, 0.6]],
    metadatas=[{"name":"Abhishek"}, {"name":"Raj"}],
    
)

In [8]:
print("Available collections:", chroma_client.list_collections())

Available collections: [Collection(name=my_collection1), Collection(name=my_collection)]


In [13]:
print("Fetching Data with IDs:", collection.get(ids=[ "1"]))

Fetching Data with IDs: {'ids': ['1'], 'embeddings': None, 'documents': [None], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'name': 'Abhishek'}]}


In [14]:
print("Fetching Data with IDs:", collection.get(ids=[ "1"], include=["embeddings","metadatas"]))

Fetching Data with IDs: {'ids': ['1'], 'embeddings': array([[0.1       , 0.2       , 0.30000001]]), 'documents': None, 'uris': None, 'included': ['embeddings', 'metadatas'], 'data': None, 'metadatas': [{'name': 'Abhishek'}]}


In [ ]:
collection.add(
    ids=["4"],
    embeddings=[[0.7, 0.8, 0.9]],
    documents=["I am learning Machine Learning"]
)

In [ ]:
results = collection.query(
    query_embeddings=[[0.1, 0.2, 0.3]],
    n_results=2 # how many results to return; number of nearest neighbors
)
print(results)

{'ids': [['1', '2']], 'embeddings': None, 'documents': [[None, None]], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'name': 'Abhishek'}, {'name': 'Raj'}]], 'distances': [[0.0, 0.27000001072883606]]}


In [ ]:
collection.update(
    ids=["2"],
    embeddings=[[0.4, 0.5, 0.9]],
    metadatas=[{"name":"Dev"}],
    
)

print("Fetching Data with IDs after update:", collection.get(ids=[ "2"], include=["embeddings","metadatas"]))

Fetching Data with IDs after update: {'ids': ['2'], 'embeddings': array([[0.40000001, 0.5       , 0.89999998]]), 'documents': None, 'uris': None, 'included': ['embeddings', 'metadatas'], 'data': None, 'metadatas': [{'name': 'Dev'}]}


In [ ]:
collection.delete(ids=["1"])
print("Fetching Data with IDs after deletion:", collection.get(ids=[ "1"]))

Fetching Data with IDs after deletion: {'ids': [], 'embeddings': None, 'documents': [], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': []}


In [ ]:
for collection in chroma_client.list_collections():
    chroma_client.delete_collection(name=collection.name)

In [ ]:
print("Available clients:", chroma_client.list_collections())

Available clients: [Collection(name=my_collection1)]


In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os
from openai import OpenAI

# openai_client = OpenAI(
#     api_key=os.getenv("OPENAI_API_KEY")
# )

In [ ]:
from sentence_transformers import SentenceTransformer

# Free, runs locally
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

def get_local_embedding(text):
    return embedding_model.encode(text).tolist()


In [ ]:
documents = [
    "The Eiffel Tower is located in Paris.",
    "The Colosseum is in Rome, Italy.",
    "The Taj Mahal is a famous monument in India.",
    "Mount Everest is the highest mountain in the world.",
    "Python is a popular programming language."
]

# Convert documents to embeddings
embeddings = [get_local_embedding(doc) for doc in documents]

# Insert into ChromaDB
collection.add(
    ids=[str(i) for i in range(len(documents))],  # Unique IDs
    documents=documents,
    embeddings=embeddings
)

print("Data added successfully!")

print("all collection data:", collection.get())


Data added successfully!
all collection data: {'ids': ['0', '1', '2', '3', '4'], 'embeddings': None, 'documents': ['The Eiffel Tower is located in Paris.', 'The Colosseum is in Rome, Italy.', 'The Taj Mahal is a famous monument in India.', 'Mount Everest is the highest mountain in the world.', 'Python is a popular programming language.'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [None, None, None, None, None]}


In [ ]:
query_text = "Where is the Eiffel Tower?"
query_embedding = get_local_embedding(query_text)

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=2,  # Get top 2 matches
    include=["documents", "distances"]
)

print("Query:", query_text)
print("Most Similar Result:", results["documents"][0])
print("Distance:", results["distances"][0])




updated_text = "The Eiffel Tower is one of the most visited landmarks in the world."
updated_embedding = get_local_embedding(updated_text)

collection.update(
    ids=["0"],  # ID of the document to update
    documents=[updated_text],
    embeddings=[updated_embedding]
)
 
print("Data updated successfully!")

Query: Where is the Eiffel Tower?
Most Similar Result: ['The Eiffel Tower is one of the most visited landmarks in the world.', 'The Colosseum is in Rome, Italy.']
Distance: [0.4766351878643036, 1.5537118911743164]
Data updated successfully!


In [ ]:
query_text = "Where is the Eiffel Tower?"
query_embedding = get_local_embedding(query_text)

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=2,  # Get top 2 matches
    include=["documents", "distances"]
)

print("Query:", query_text)
print("Most Similar Result:", results["documents"][0])
print("Distance:", results["distances"][0])





Query: Where is the Eiffel Tower?
Most Similar Result: ['The Eiffel Tower is one of the most visited landmarks in the world.', 'The Colosseum is in Rome, Italy.']
Distance: [0.4766351878643036, 1.5537118911743164]


In [ ]:
ht_tower_text="Eiffel Tower is 330m tall"

collection.add(
    ids=["6"],
    embeddings=get_local_embedding(ht_tower_text),
    documents=ht_tower_text
)

In [ ]:
query_text = "Where is the Eiffel Tower?"
query_embedding = get_local_embedding(query_text)

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=2,  # Get top 2 matches
    include=["documents", "distances"]
)

print("Query:", query_text)
print("Most Similar Result:", results["documents"][0])
print("Distance:", results["distances"][0])



Query: Where is the Eiffel Tower?
Most Similar Result: ['Eiffel Tower is 330m tall', 'The Colosseum is in Rome, Italy.']
Distance: [0.3831125497817993, 1.5537118911743164]


In [ ]:
collection.delete(ids=["0"])
print("Data with ID '0' deleted successfully!")
print("All collection data after deletion:", collection.get())

Data with ID '0' deleted successfully!
All collection data after deletion: {'ids': ['1', '2', '3', '4', '6'], 'embeddings': None, 'documents': ['The Colosseum is in Rome, Italy.', 'The Taj Mahal is a famous monument in India.', 'Mount Everest is the highest mountain in the world.', 'Python is a popular programming language.', 'Eiffel Tower is 330m tall'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [None, None, None, None, None]}


In [ ]:
query_text = "Where is the Eiffel Tower?"
query_embedding = get_local_embedding(query_text)

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=2,  # Get top 2 matches
    include=["documents", "distances"]
)

print("Query:", query_text)
print("Most Similar Result:", results["documents"][0])
print("Distance:", results["distances"][0])

Query: Where is the Eiffel Tower?
Most Similar Result: ['Eiffel Tower is 330m tall', 'The Colosseum is in Rome, Italy.']
Distance: [0.3831125497817993, 1.5537118911743164]
